# OOP Week 12 -- Plugin Exercise

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-11
**Focus:** add a new analyzer WITHOUT touching core code

---

## Learning Objectives

1. Add a new component to an existing system without modification
2. Demonstrate OCP in practice
3. Write tests for the new component
4. Register the component with the factory
5. Verify the pipeline still works

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## The Challenge

You will add a **PercentileAnalyzer** to the pipeline. Rules:

1. Do NOT modify any existing class (AnalyzerBase, MeanAnalyzer, etc.)
2. Do NOT modify the AnalyzerFactory class
3. Do NOT modify any existing test
4. Write the new analyzer in a new class
5. Register it with the factory (ONE line)
6. Write at least 3 tests for it

This proves that the architecture is truly open for extension.

---
## Existing Code (DO NOT MODIFY)

In [ ]:
# ===== EXISTING CODE -- DO NOT MODIFY =====

class AnalyzerBase:
    def analyze(self, values):
        raise NotImplementedError

class MeanAnalyzer(AnalyzerBase):
    def analyze(self, values):
        return {"mean": round(sum(values)/len(values), 4)} if values else {}

class StdAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values)/len(values)
        return {"std": round((sum((x-m)**2 for x in values)/len(values))**0.5, 4)}

class EventAnalyzer(AnalyzerBase):
    def __init__(self, threshold=50):
        self.threshold = threshold
    def analyze(self, values):
        return {"events_above": sum(1 for v in values if v > self.threshold)}

class AnalyzerFactory:
    _registry = {}
    @classmethod
    def register(cls, name, klass):
        cls._registry[name] = klass
    @classmethod
    def create(cls, name, **kw):
        if name not in cls._registry:
            raise ValueError("Unknown: " + name)
        return cls._registry[name](**kw)
    @classmethod
    def list_available(cls):
        return list(cls._registry.keys())

# Register existing
AnalyzerFactory.register("mean", MeanAnalyzer)
AnalyzerFactory.register("std", StdAnalyzer)
AnalyzerFactory.register("events", EventAnalyzer)

print("Existing analyzers:", AnalyzerFactory.list_available())
# ===== END EXISTING CODE =====

**Expected Output:**
```
Existing analyzers: ['mean', 'std', 'events']
```

---
## Your Task: Add PercentileAnalyzer

Create a `PercentileAnalyzer` class that:
- Inherits from `AnalyzerBase`
- Takes a list of percentiles (default: [25, 50, 75])
- Returns dict with keys like `p25`, `p50`, `p75`
- Handles empty input gracefully

In [ ]:
# Step 1: Create the class (inherits from AnalyzerBase)
# YOUR CODE HERE

# Step 2: Register with factory (ONE line)
# YOUR CODE HERE

# Step 3: Test it
# YOUR CODE HERE

---
## Solution (try yourself first!)

In [ ]:
class PercentileAnalyzer(AnalyzerBase):
    """Computes percentiles of numeric values."""
    def __init__(self, percentiles=None):
        self.percentiles = percentiles or [25, 50, 75]

    def analyze(self, values):
        if not values:
            return {}
        s = sorted(values)
        n = len(s)
        result = {}
        for p in self.percentiles:
            idx = min(int(n * p / 100), n - 1)
            result["p" + str(p)] = s[idx]
        return result

# Register (ONE line!)
AnalyzerFactory.register("percentile", PercentileAnalyzer)

print("Available:", AnalyzerFactory.list_available())

# Create from factory
pa = AnalyzerFactory.create("percentile", percentiles=[10, 50, 90])
values = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
print("Results:", pa.analyze(values))

**Expected Output:**
```
Available: ['mean', 'std', 'events', 'percentile']
Results: {'p10': 20, 'p50': 60, 'p90': 100}
```

---
## Tests for PercentileAnalyzer

In [ ]:
def test_percentile_normal():
    pa = PercentileAnalyzer()
    result = pa.analyze([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    assert "p25" in result
    assert "p50" in result
    assert "p75" in result

def test_percentile_empty():
    pa = PercentileAnalyzer()
    assert pa.analyze([]) == {}

def test_percentile_custom():
    pa = PercentileAnalyzer(percentiles=[50])
    result = pa.analyze([1, 2, 3, 4, 5])
    assert "p50" in result
    assert "p25" not in result  # only asked for 50

def test_percentile_factory():
    pa = AnalyzerFactory.create("percentile")
    assert isinstance(pa, PercentileAnalyzer)

test_percentile_normal()
print("[PASS] test_percentile_normal")
test_percentile_empty()
print("[PASS] test_percentile_empty")
test_percentile_custom()
print("[PASS] test_percentile_custom")
test_percentile_factory()
print("[PASS] test_percentile_factory")
print()
print("All plugin tests passed!")
print("And we did NOT modify any existing code!")

**Expected Output:**
```
[PASS] test_percentile_normal
[PASS] test_percentile_empty
[PASS] test_percentile_custom
[PASS] test_percentile_factory

All plugin tests passed!
And we did NOT modify any existing code!
```

---
## Section 3: Your Turn -- Second Plugin

Now add a SECOND plugin: a `VarianceAnalyzer`.

In [ ]:
# YOUR TASK: Create VarianceAnalyzer
# 1. Inherit from AnalyzerBase
# 2. Implement analyze(values) -> {"variance": ...}
# 3. Register with factory
# 4. Write 3 tests

# YOUR CODE HERE


### Solution

In [ ]:
class VarianceAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values) / len(values)
        var = sum((x - m) ** 2 for x in values) / len(values)
        return {"variance": round(var, 4)}

AnalyzerFactory.register("variance", VarianceAnalyzer)

# Tests
def test_variance_normal():
    va = VarianceAnalyzer()
    r = va.analyze([10, 20, 30])
    assert "variance" in r
    assert r["variance"] > 0

def test_variance_empty():
    assert VarianceAnalyzer().analyze([]) == {}

def test_variance_uniform():
    r = VarianceAnalyzer().analyze([5, 5, 5])
    assert r["variance"] == 0

test_variance_normal()
print("[PASS] test_variance_normal")
test_variance_empty()
print("[PASS] test_variance_empty")
test_variance_uniform()
print("[PASS] test_variance_uniform")
print()
print("Available:", AnalyzerFactory.list_available())

**Expected Output:**
```
[PASS] test_variance_normal
[PASS] test_variance_empty
[PASS] test_variance_uniform

Available: ['mean', 'std', 'events', 'percentile', 'variance']
```

---
## Section 4: Full Pipeline with Plugins

In [ ]:
# Run ALL registered analyzers on sample data
values = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

print("=== Running all registered analyzers ===")
all_results = {}
for name in AnalyzerFactory.list_available():
    try:
        a = AnalyzerFactory.create(name)
        result = a.analyze(values)
        all_results[name] = result
        print(name + ": " + str(result))
    except TypeError:
        # Some analyzers need params (like EventAnalyzer)
        print(name + ": (needs parameters, skipping)")

print()
print("Total analyzers:", len(AnalyzerFactory.list_available()))
print("All added without modifying core!")

**Expected Output:**
```
=== Running all registered analyzers ===
mean: {'mean': 55.0}
std: {'std': 28.7228}
events: {'events_above': 5}
percentile: {'p25': 30, 'p50': 60, 'p75': 80}
variance: {'variance': 825.0}

Total analyzers: 5
All added without modifying core!
```

---
### Try It!

Create a third plugin: `MedianAnalyzer`. Register it, test it, and verify the total count increases.

In [ ]:
# YOUR CODE HERE


---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Starting with the AnalyzerFactory from today, add TWO new plugins: MovingAverageAnalyzer and ZScoreAnalyzer. Write tests for both. Do NOT modify existing code.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# How does the Plugin Exercise prove that our architecture follows OCP (Week 7)?

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Think about browser extensions. How are they 'plugins'? What interface do they implement?

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Plugin architecture** | A system where new features are added without modifying core |
| **Extension point** | A place in the code where plugins can hook in |
| **Registration** | Adding a new class to the factory/registry |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: Which principle did we demonstrate? (SRP, OCP, or both?)
# Answer: 

# Q2: How many existing files did we modify?
# Answer: 

# Q3: What made this possible? (what patterns?)
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)